In [32]:
import logging
import warnings

In [33]:
warnings.filterwarnings('ignore')

In [34]:
import pandas as pd

In [35]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [121]:
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential, load_model
from keras.layers import Dense, Dropout
from keras.callbacks import EarlyStopping

In [122]:
import joblib

In [141]:
import numpy as np

In [185]:
class IAModel:
    scaler_counter = 0
    def __init__(self, data, y_col):
        self.data = data
        self.y_col = y_col
        self.load_data()
        self.x_train = None
        self.x_test = None
        self.y_train = None
        self.y_test = None
        self._model = None
        IAModel.scaler_counter += 1
        return self._model

    def load_data(self):
        data = pd.read_csv(self.data)
        if data.isnull().sum().any():
            logging.info("Valeur manquante détectée...")
            logging.info("Imputation des valeurs manquantes avec la moyenne...")
            data = data.fillna(data.mean())
            logging.info("Valeur manquante gérée.")
        x = data.drop(self.y_col, axis=1)
        y = data[self.y_col]
        return x,y
        
    def train_test_split(self, test_size, random_state, shuffle=True):
        self.x_train, self.x_test, self.y_train, self.y_test = train_test_split(
            self.x, 
            self.y, 
            test_size=test_size, 
            random_state=random_state, 
            shuffle=shuffle
        )

    def data_display(self):
        print(f"{'*'*15} Donnée d'entrainement {'*'*15}")
        display(self.x_train)
        print(f"{'*'*15} Donnée de test {'*'*15}")
        display(self.x_test)
        print(f"{'='*15} SORTIE {'='*15}")
        print(f"{'*'*15} Sortie d'entrainement {'*'*15}")
        display(self.y_train)
        print(f"{'*'*15} Sortie de test {'*'*15}")   
        display(self.y_test)
        

    def normalize(self):
        scaler = StandardScaler()
        if self.x_train is not None and self.x_test is not None:
            x_train_norm = scaler.fit_transform(self.x_train)
            x_test_norm = scaler.transform(self.x_test)
            scaler_path = "scaler_" + f"000{str(IAModel.scaler_counter)}.save"
            logging.info("Sauvegarde en cours...")
            joblib.dump(scaler, scaler_path)
            logging.info("Scaler sauvegardé")
            return x_train_norm, x_test_norm
        logging.error("Valeurs d'entrainement et de test invalides.")
        return None

    def contruct(self, layers:list):
        self._model = Sequential(layers)
        return self._model

    def compile(self, optimizer, loss, metrics:list):
        if self._model == None:
            return False
        self._model.compile(optimizer=optimizer, loss=loss, metrics=metrics)

    def summary(self):
        if self._model == None:
            return 
        self._model.summary()

    def fit(self, epochs, batch_size, validation_split,verbose=1, callbacks = []):
        self._model.fit(
            self.x_train_norm,
            self.y_train,
            epochs=epochs,
            batch_size=batch_size,
            callbacks=callbacks,
            verbose=verbose
        )
        return self._model

    def evaluate(self, verbose=1):
        if self._model == None:
            logging.error("Le modèle est vide.")
            return
        if self.x_test_norm is None and self.y_test is None:
            logging.error("Valeurs de test invalides.")
            return 
        self._model.evaluate(self.x_test_norm, self.y_test, verbose=verbose)
        return self._model

    @staticmethod
    def layer(units, activation,**kargs):
        return Dense(units, activation=activation,**kargs)

    @staticmethod
    def dropout(size:float):
        return Dropout(size)
        
    def early_stopping(self, patience, restore_best_weights=True, **kargs):
        return EarlyStopping(
            patience=patience,
            restore_best_weights=True,
            **kargs
        )

    def predict_form_data_test(self, flatten=True):
        if self._model is None:
            return
        if flatten:
            return self._model.predict(self.x_test_norm).flatten()
        return self._model.predict(self.x_test_norm)

    def predict(self, data):
        if self._model is None:
            return
        scaler = joblib.load(f"scaler_000{IAModel.scaler_counter}.save")
        data = np.array([data])
        data_norm = scaler.transform(data)
        return self._model.predict(data_norm)[0][0]

    def save(self, filename):
        if self._model is None:
            logging.error("Le modèle est vide")
            return 
        self._model.save(f"{filename}.keras")
    
    def load_model(self, path):
        self._model = load_model(path)
        return self._model

    @property
    def x(self):
        return self.load_data()[0]
        
    @property
    def y(self):
        return self.load_data()[1]

    @property
    def x_train_norm(self):
        return self.normalize()[0]

    @property
    def x_test_norm(self):
        return self.normalize()[1]

In [186]:
model = IAModel('data/dataset_etudiants_600.csv', 'note_exam')

In [222]:
model.train_test_split(0.2,42)
model.normalize()
model.contruct([
    IAModel.layer(units=10, activation='relu', name='layer_01', input_shape=(model.x_train_norm.shape[1],)),
    IAModel.dropout(0.2),
    IAModel.layer(units=30, activation='relu', name='layer_02'),
    IAModel.dropout(0.2),
    IAModel.layer(units=10, activation='relu', name='layer_03'),
    IAModel.layer(1,activation='relu')
])
model.compile(optimizer='adam', loss='mse', metrics=['mae', 'mse'])

In [223]:
model.fit(
    epochs=800, 
    batch_size=42, 
    validation_split=0.2, 
    verbose=0, 
    callbacks=[model.early_stopping(patience=20, verbose=1)])

<Sequential name=sequential_36, built=True>

In [196]:
model.evaluate()

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 42.8121 - mae: 5.6334 - mse: 42.8121


<Sequential name=sequential_35, built=True>

In [197]:
model.save('model_04')

In [220]:
print(model.predict([40,100,3,4,1]))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step
12.300046


In [221]:
new_model = model.load_model('model_04.keras')